In [ ]:
import requests
import json



headers = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28"
}

def create_page(title: str, content: str):
    url = "https://api.notion.com/v1/pages"

    data = {
        "parent": {"database_id": DATABASE_ID},
        "properties": {
            "이름": {
                "title": [{"text": {"content": title}}]
            }
        },
        "children": [
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {
                    "rich_text": [
                        {"type": "text", "text": {"content": content}}
                    ]
                }
            }
        ]
    }

    res = requests.post(url, headers=headers, data=json.dumps(data))
    print(res.status_code, res.text)

create_page("테스트 글", "파이썬으로 노션 글을 작성했습니다!")


200 {"object":"page","id":"2ba77d75-e008-8118-944e-e1ae29052596","created_time":"2025-11-29T16:07:00.000Z","last_edited_time":"2025-11-29T16:07:00.000Z","created_by":{"object":"user","id":"27f33389-3fc2-4970-a7ed-956896196505"},"last_edited_by":{"object":"user","id":"27f33389-3fc2-4970-a7ed-956896196505"},"cover":null,"icon":null,"parent":{"type":"database_id","database_id":"2ba77d75-e008-80df-ba06-df4548ddc834"},"archived":false,"in_trash":false,"is_locked":false,"properties":{"이름":{"id":"title","type":"title","title":[{"type":"text","text":{"content":"테스트 글","link":null},"annotations":{"bold":false,"italic":false,"strikethrough":false,"underline":false,"code":false,"color":"default"},"plain_text":"테스트 글","href":null}]}},"url":"https://www.notion.so/2ba77d75e0088118944ee1ae29052596","public_url":null,"request_id":"c2c7aff5-0be7-4108-8fff-2949908c318a"}


예시 코드 및 논문: Paper2code
git: https://github.com/going-doer/Paper2Code.git
paper method text:
"3 METHOD In this section, we start with describing the task of repository-level code generation from machine learning papers, and propose PaperCoder, a multi-agent, multi-stage framework designed to tackle it. 3.1 REPOSITORY-LEVEL CODE GENERATION FROM MACHINE LEARNING PAPERS The goal of our repository-level code generation task is to automatically produce a repository that faithfully implements methods and experiments described in machine learning papers (especially for cases where authors do not release their code), to support reproducibility and accelerate scientific progress (Pineau et al., 2021; Magnusson et al., 2023). Formally, we define this task as a function (or a model) M that maps a paper R to a corresponding code repository C, as follows: M(R) = C. Here, C is composed of multiple files {c1, c2, ..., cn}, each responsible for implementing different components of the methods and experiments in R, but together they should form a cohesive pipeline. The most straightforward approach to instantiating M is to instruct the LLM to generate the entire code repository, conditioned on the given paper, as follows: M(R) := LLM(T (R)), where T is the prompt template that specifies the intended behavior of the LLM for the target task (including task descriptions, detailed instructions, and any other relevant context). Yet, generating a complete, modular, and faithful repository in a single pass is extremely challenging, even for powerful LLMs, due to the inherent complexity of scientific papers and their corresponding implementations, the longcontext limitations of current models, and the difficulty in maintaining consistent global structure and cross-file dependencies. Therefore, we propose to decompose the overall task into smaller subtasks, each handled by a specialized agent tailored to a specific aspect of paper-to-code transformation. 3.2 PAPERCODER: LLM-POWERED MULTI-AGENT FRAMEWORK FOR PAPER-TO-CODE We now introduce PaperCoder, a structured, multi-agent framework for generating code repositories directly from machine learning papers (without access to pre-existing artifacts or implementations, such as skeleton code). Specifically, inspired by typical software development workflows, PaperCoder decomposes the task into three coordinated stages: Planning, Analysis, and Coding, each orchestrated by specialized LLM agents. Formally, given a paper R, the overall process can be defined as follows: Planning: P = Mplan(R), Analysis: A = Manalysis(R, P), Coding: C = Mcode(R, P, A), where P, A, and C represent the high-level implementation plan, the detailed function-level analysis, and the final code repository, respectively. The overall pipeline of PaperCoder is shown in Figure 2.
Figure 2: (Left) The naive approach, which directly generates an entire code repository from a paper. (Right) Our PaperCoder framework, which is operationalized by decomposing the task into three stages: (1) Planning, where a high-level implementation plan is constructed from the paper, including overall plan, architectural design, logic design, and configuration file; (2) Analysis, where the plan is translated into detailed file-level specifications; and (3) Coding, where the final codes are generated to implement the methods and experiments of the paper. 3.2.1 PLANNING It is worth noting that, in contrast to implementation specifications designed explicitly for software development, papers are written to communicate ideas and findings to humans. As a result, they often contain high-level motivations, persuasive narratives, and auxiliary details that are crucial for human understanding but noisy, loosely specified, or ambiguous from a software engineering perspective. To mitigate this, we introduce a planning phase that transforms unstructured textual content into implementation-level abstractions. Also, we decompose the planning process into four sequential subcomponents (to simplify the task and reduce cognitive load of LLM-powered agents at each step): 1) overall plan, 2) architecture design, 3) logic design, and 4) configuration generation. Formally, we define this as: Mplan(R) → P = {o, d, l, g}, where o is the overall plan, d is the architecture design, l is the logic design, and g is the configuration file, with each stage using the outputs of the previous ones as contextual input. We then describe how each subcomponent is instantiated below. Overall Plan The first step is to extract a high-level summary of the core components and functionalities described throughout the paper, to identify the specific methods and experiments to be implemented. In other words, this high-level overview includes model components, training objectives, data processing steps, and evaluation protocols (distributed across the entire paper), which can form the foundation for all subsequent steps, formalized as follows: M (1) plan(R) := LLM(T (1) plan (R)) → o. Architecture Design Based on the extracted overall plan alongside the input paper, the next step is to define the repository-level architecture, which includes identifying files, organizing them into modules, and defining their relationships, to ensure a coherent and maintainable structure. Specifically, the LLM-powered agent is prompted to generate a file list, which outlines the overall file structure of the repository; a class diagram, which details static representations of files (such as core classes and their attributes); and a sequence diagram, which models the dynamic interactions. Formally, similar to overall plan, this process can be defined as follows: M (2) plan(R, o) := LLM(T (2) plan (R, o)) → d. Logic Design While the previous architecture design focuses on what to build, the logic design phase specifies how these components should be instantiated in practice by considering their dependencies in terms of overall execution flow. This step is crucial because individual modules often depend on shared utilities, configurations, or data loaders that are defined in other parts of the repository, and without an explicitly defined execution order, the code generation can result in failure or inconsistency (e.g., generating file B before file A when B imports modules from A). To address this, the logic design stage not only produces an ordered file list that dictates the sequence in which the files should be implemented and executed, but also further elaborates on the logic within each file; thereby, providing more fine-grained specifications. Formally, M (3) plan(R, o, d) := LLM(T (3) plan (R, o, d)) → l. Configuration Generation In the last stage of planning, PaperCoder synthesizes a configuration file (config.yaml) that includes key hyperparameters, model settings, and other runtime options based on prior outputs alongside the given paper. We note that, in addition to grounding the code generation process with the explicit configuration details, it enables researchers to easily review and adjust experimental configurations without modifying the source code. Formally, M (4) plan(R, o, d, l) := LLM(T (4) plan (R, o, d, l)) → g. We provide prompts used to elicit each planning output in Appendix D. 3.2.2 ANALYSIS Following the planning stage, which defines the overall structure and execution flow of the repository, the analysis phase focuses on interpreting and specifying the implementation-level details for modules within each file. In other words, unlike planning that answers what components to build and how they relate, this phase addresses the question of how each component should be operationalized and concretely implemented at the file level, which includes the definition of functional goals, input-output behaviors, intra- and inter-file dependencies, and algorithmic specifications derived from the original paper. Specifically, given the input paper R and planning outputs P = {o, d, l, g}, the analysis agent iteratively processes each file fi (identified during planning) and generates a detailed analysis ai describing what needs to be implemented in that file. Formally, {Manalysis(R, P, fi)} n=|F | i=1 where Manalysis(R, P, fi) := LLM(Tanalysis(R, P, fi)) → ai , with F as the set of identified files, e.g., fi ∈ F. 3.2.3 CODING The final stage is the coding phase, where the complete code repository is produced. In particular, each file is generated based on all the available contextual information accumulated from the previous stages, including the overall plan, architecture design, logic design, configuration file, and file-specific analyses, as well as the original paper. Additionally, to ensure consistency across different files, we generate them sequentially according to the execution order (i.e., the ordered file list determined during the logic design stage). To be formal, for each file fi , the corresponding code ci is generated as follows: Mcode(R, P, fi , ai , {c1, ..., ci−1}) := LLM(Tcode(R, P, fi , ai , {c1, ..., ci−1})) → ci , resulting in the complete code repository C = {ci} n=|F | i=1 . We note that this iterative formulation can ensure that i-th code is generated with full awareness of its dependencies and the evolving state of the repository."

In [2]:
!pip install pyautogen gitpython openai tiktoken


  Using cached pyautogen-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached gitpython-3.1.45-py3-none-any.whl.metadata (13 kB)
  Using cached openai-2.8.1-py3-none-any.whl.metadata (29 kB)
  Using cached tiktoken-0.12.0-cp39-cp39-win_amd64.whl.metadata (6.9 kB)
  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached asyncer-0.0.8-py3-none-any.whl.metadata (6.7 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydan

In [ ]:
import os
import ast
import json
import tempfile
from git import Repo
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from openai import OpenAI

##########################
# OpenAI 설정
##########################
# client = OpenAI(
#     api_key=os.environ[""],
#     base_url="https://openrouter.ai/api/v1"
# )
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

##########################
# Embedding Helper
##########################
def embed_text(text: str):
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return emb.data[0].embedding


def cosine_similarity(a, b):
    import numpy as np
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


##########################
# Repo Agent Tools
##########################

def clone_repo(repo_url):
    temp_dir = tempfile.mkdtemp()
    Repo.clone_from(repo_url, temp_dir)
    return temp_dir


def list_python_files(repo_path):
    py_files = []
    for root, dirs, files in os.walk(repo_path):
        for f in files:
            if f.endswith(".py"):
                full = os.path.join(root, f)
                py_files.append(full)
    return py_files


def parse_ast(path):
    with open(path, "r", encoding="utf-8") as f:
        code = f.read()

    tree = ast.parse(code)

    functions = []
    classes = []

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            src = ast.get_source_segment(code, node)
            functions.append({"name": node.name, "source": src})
        elif isinstance(node, ast.ClassDef):
            src = ast.get_source_segment(code, node)
            classes.append({"name": node.name, "source": src})

    return code, functions, classes


############################################
# 1) Repo Agent
############################################

repo_agent_prompt = """
You are RepoAgent.

Your tasks:
- Clone GitHub repo
- List all .py files
- Parse AST for each file
- Return {file_path, source, functions, classes}

DO NOT generate analysis or summary. Only file/AST extraction.
"""

repo_agent = AssistantAgent(
    name="repo_agent",
    system_message=repo_agent_prompt
)

############################################
# 2) Code Analysis Agent
############################################

code_agent_prompt = """
You are CodeAnalysisAgent.

Given Python file source + function list:
- Provide high-level semantic explanation of each function.
- No similarity comparison yet.
"""

code_agent = AssistantAgent(
    name="code_analysis_agent",
    system_message=code_agent_prompt
)

############################################
# 3) Paper Matching Agent
############################################

paper_matching_prompt = """
You are PaperMatchingAgent.

For each function:
- Compute similarity between function source and paper text
- Identify the most relevant sentence in the paper text
- Return similarity + matched_text
"""

paper_agent = AssistantAgent(
    name="paper_matching_agent",
    system_message=paper_matching_prompt
)

############################################
# 4) Summary Agent
############################################

summary_agent_prompt = """
You are SummaryAgent.

Combine all results and produce final JSON:

{
 "reference_code_analysis": {
   "files": [
     {
       "file_name": "",
       "file_summary": "",
       "functions": [
         {
           "name": "",
           "summary": "",
           "similarity": 0.xx,
           "matched_text": ""
         }
       ]
     }
   ]
 }
}

Only output JSON. Nothing else.
"""

summary_agent = AssistantAgent(
    name="summary_agent",
    system_message=summary_agent_prompt
)

############################################
# 5) Orchestrator Agent
############################################

orch_prompt = """
You are the OrchestratorAgent.

Steps:
1. Ask RepoAgent to clone and parse the repo.
2. For each file, send functions to CodeAnalysisAgent.
3. For each function analysis, send to PaperMatchingAgent.
4. Collect all results and send to SummaryAgent.
5. Return final JSON to user.

Do not do analysis yourself. Delegate everything.
"""

orchestrator = AssistantAgent(
    name="orchestrator",
    system_message=orch_prompt
)

############################################
# GROUP CHAT (Multi-Agent)
############################################

groupchat = GroupChat(
    agents=[orchestrator, repo_agent, code_agent, paper_agent, summary_agent],
    messages=[]
)

manager = GroupChatManager(groupchat=groupchat)

############################################
# 실행 함수
############################################

def run_agent1(github_url, paper_text):
    # 1) Clone repo
    repo_path = clone_repo(github_url)
    py_files = list_python_files(repo_path)

    file_results = []

    for file_path in py_files:
        source, functions, classes = parse_ast(file_path)

        # CodeAnalysis
        analysis_results = []
        for fn in functions:
            msg = {
                "file_path": file_path,
                "function_name": fn["name"],
                "source": fn["source"]
            }
            analysis_response = manager.run(
                assistant="code_analysis_agent",
                message=str(msg)
            )
            analysis_text = analysis_response["messages"][-1]["content"]
            analysis_results.append({
                "name": fn["name"],
                "analysis": analysis_text
            })

        # Paper Similarity
        similarity_results = []
        paper_emb = embed_text(paper_text)

        for fn in functions:
            func_emb = embed_text(fn["source"])
            sim = cosine_similarity(func_emb, paper_emb)

            similarity_results.append({
                "name": fn["name"],
                "similarity": sim,
                "matched_text": "matched part from paper"
            })

        file_results.append({
            "file_name": os.path.basename(file_path),
            "file_summary": "Semantic summary of {}".format(file_path),
            "functions": similarity_results
        })

    # Final JSON
    final_msg = manager.run(
        assistant="summary_agent",
        message=json.dumps({"files": file_results}, ensure_ascii=False)
    )

    return final_msg


############################################
# MAIN EXECUTION
############################################

if __name__ == "__main__":

    github_url = "https://github.com/going-doer/Paper2Code.git"
    paper_text = "3 METHOD In this section, we start with describing the task of repository-level code generation from machine learning papers, and propose PaperCoder, a multi-agent, multi-stage framework designed to tackle it. 3.1 REPOSITORY-LEVEL CODE GENERATION FROM MACHINE LEARNING PAPERS The goal of our repository-level code generation task is to automatically produce a repository that faithfully implements methods and experiments described in machine learning papers (especially for cases where authors do not release their code), to support reproducibility and accelerate scientific progress (Pineau et al., 2021; Magnusson et al., 2023). Formally, we define this task as a function (or a model) M that maps a paper R to a corresponding code repository C, as follows: M(R) = C. Here, C is composed of multiple files {c1, c2, ..., cn}, each responsible for implementing different components of the methods and experiments in R, but together they should form a cohesive pipeline. The most straightforward approach to instantiating M is to instruct the LLM to generate the entire code repository, conditioned on the given paper, as follows: M(R) := LLM(T (R)), where T is the prompt template that specifies the intended behavior of the LLM for the target task (including task descriptions, detailed instructions, and any other relevant context). Yet, generating a complete, modular, and faithful repository in a single pass is extremely challenging, even for powerful LLMs, due to the inherent complexity of scientific papers and their corresponding implementations, the longcontext limitations of current models, and the difficulty in maintaining consistent global structure and cross-file dependencies. Therefore, we propose to decompose the overall task into smaller subtasks, each handled by a specialized agent tailored to a specific aspect of paper-to-code transformation. 3.2 PAPERCODER: LLM-POWERED MULTI-AGENT FRAMEWORK FOR PAPER-TO-CODE We now introduce PaperCoder, a structured, multi-agent framework for generating code repositories directly from machine learning papers (without access to pre-existing artifacts or implementations, such as skeleton code). Specifically, inspired by typical software development workflows, PaperCoder decomposes the task into three coordinated stages: Planning, Analysis, and Coding, each orchestrated by specialized LLM agents. Formally, given a paper R, the overall process can be defined as follows: Planning: P = Mplan(R), Analysis: A = Manalysis(R, P), Coding: C = Mcode(R, P, A), where P, A, and C represent the high-level implementation plan, the detailed function-level analysis, and the final code repository, respectively. The overall pipeline of PaperCoder is shown in Figure 2. Figure 2: (Left) The naive approach, which directly generates an entire code repository from a paper. (Right) Our PaperCoder framework, which is operationalized by decomposing the task into three stages: (1) Planning, where a high-level implementation plan is constructed from the paper, including overall plan, architectural design, logic design, and configuration file; (2) Analysis, where the plan is translated into detailed file-level specifications; and (3) Coding, where the final codes are generated to implement the methods and experiments of the paper. 3.2.1 PLANNING It is worth noting that, in contrast to implementation specifications designed explicitly for software development, papers are written to communicate ideas and findings to humans. As a result, they often contain high-level motivations, persuasive narratives, and auxiliary details that are crucial for human understanding but noisy, loosely specified, or ambiguous from a software engineering perspective. To mitigate this, we introduce a planning phase that transforms unstructured textual content into implementation-level abstractions. Also, we decompose the planning process into four sequential subcomponents (to simplify the task and reduce cognitive load of LLM-powered agents at each step): 1) overall plan, 2) architecture design, 3) logic design, and 4) configuration generation. Formally, we define this as: Mplan(R) → P = {o, d, l, g}, where o is the overall plan, d is the architecture design, l is the logic design, and g is the configuration file, with each stage using the outputs of the previous ones as contextual input. We then describe how each subcomponent is instantiated below. Overall Plan The first step is to extract a high-level summary of the core components and functionalities described throughout the paper, to identify the specific methods and experiments to be implemented. In other words, this high-level overview includes model components, training objectives, data processing steps, and evaluation protocols (distributed across the entire paper), which can form the foundation for all subsequent steps, formalized as follows: M (1) plan(R) := LLM(T (1) plan (R)) → o. Architecture Design Based on the extracted overall plan alongside the input paper, the next step is to define the repository-level architecture, which includes identifying files, organizing them into modules, and defining their relationships, to ensure a coherent and maintainable structure. Specifically, the LLM-powered agent is prompted to generate a file list, which outlines the overall file structure of the repository; a class diagram, which details static representations of files (such as core classes and their attributes); and a sequence diagram, which models the dynamic interactions. Formally, similar to overall plan, this process can be defined as follows: M (2) plan(R, o) := LLM(T (2) plan (R, o)) → d. Logic Design While the previous architecture design focuses on what to build, the logic design phase specifies how these components should be instantiated in practice by considering their dependencies in terms of overall execution flow. This step is crucial because individual modules often depend on shared utilities, configurations, or data loaders that are defined in other parts of the repository, and without an explicitly defined execution order, the code generation can result in failure or inconsistency (e.g., generating file B before file A when B imports modules from A). To address this, the logic design stage not only produces an ordered file list that dictates the sequence in which the files should be implemented and executed, but also further elaborates on the logic within each file; thereby, providing more fine-grained specifications. Formally, M (3) plan(R, o, d) := LLM(T (3) plan (R, o, d)) → l. Configuration Generation In the last stage of planning, PaperCoder synthesizes a configuration file (config.yaml) that includes key hyperparameters, model settings, and other runtime options based on prior outputs alongside the given paper. We note that, in addition to grounding the code generation process with the explicit configuration details, it enables researchers to easily review and adjust experimental configurations without modifying the source code. Formally, M (4) plan(R, o, d, l) := LLM(T (4) plan (R, o, d, l)) → g. We provide prompts used to elicit each planning output in Appendix D. 3.2.2 ANALYSIS Following the planning stage, which defines the overall structure and execution flow of the repository, the analysis phase focuses on interpreting and specifying the implementation-level details for modules within each file. In other words, unlike planning that answers what components to build and how they relate, this phase addresses the question of how each component should be operationalized and concretely implemented at the file level, which includes the definition of functional goals, input-output behaviors, intra- and inter-file dependencies, and algorithmic specifications derived from the original paper. Specifically, given the input paper R and planning outputs P = {o, d, l, g}, the analysis agent iteratively processes each file fi (identified during planning) and generates a detailed analysis ai describing what needs to be implemented in that file. Formally, {Manalysis(R, P, fi)} n=|F | i=1 where Manalysis(R, P, fi) := LLM(Tanalysis(R, P, fi)) → ai , with F as the set of identified files, e.g., fi ∈ F. 3.2.3 CODING The final stage is the coding phase, where the complete code repository is produced. In particular, each file is generated based on all the available contextual information accumulated from the previous stages, including the overall plan, architecture design, logic design, configuration file, and file-specific analyses, as well as the original paper. Additionally, to ensure consistency across different files, we generate them sequentially according to the execution order (i.e., the ordered file list determined during the logic design stage). To be formal, for each file fi , the corresponding code ci is generated as follows: Mcode(R, P, fi , ai , {c1, ..., ci−1}) := LLM(Tcode(R, P, fi , ai , {c1, ..., ci−1})) → ci , resulting in the complete code repository C = {ci} n=|F | i=1 . We note that this iterative formulation can ensure that i-th code is generated with full awareness of its dependencies and the evolving state of the repository."  # replace

    result = run_agent1(github_url, paper_text)
    print(result)


IndexError: list index out of range